In [ ]:
import duckdb as db
from extract import get_sports, get_leagues

In [ ]:
get_sports()

In [ ]:
link_regex = r'/api/v\d(?:\.\d)?((?:/[a-zA-Z0-9]+)+)'
CREATE_sports = '''
    CREATE TABLE sports(
        sport_id INT PRIMARY KEY,
        sport_code VARCHAR(3) UNIQUE NOT NULL,
        sport_name VARCHAR(255) UNIQUE NOT NULL,
        sport_abbr VARCHAR(255) UNIQUE NOT NULL,
        sort_order INT UNIQUE NOT NULL CHECK (sort_order > 0),
        sport_link VARCHAR(255) UNIQUE NOT NULL
        ------------------------------------------------
        CONSTRAINT sport_link_format CHECK (sport_link ~ link_regex)
    )
'''
CREATE_sports = CREATE_sports.replace('link_regex', '\'' + link_regex + '\'')
print(CREATE_sports)

In [ ]:
get_leagues()

In [ ]:
CREATE_leagues = '''
    CREATE TABLE leagues(
        league_id INT PRIMARY KEY,
        sport_id INT REFERENCES sports(sport_id),
        league_name VARCHAR(255) NOT NULL,
        league_abbr VARCHAR(255) NOT NULL,
        is_active BOOLEAN NOT NULL,
        sort_order INT NOT NULL CHECK (sort_order > 0),
        league_link VARCHAR(255) UNIQUE NOT NULL
        ------------------------------------------------
        CONSTRAINT league_link_format CHECK (league_link ~ link_regex)
    )
'''
CREATE_leagues = CREATE_leagues.replace('link_regex', '\'' + link_regex + '\'')
print(CREATE_leagues)

In [ ]:
CREATE_divisions = '''
CREATE TABLE divisions(
    division_id INT PRIMARY KEY,
    league_id INT REFERENCES leagues(league_id),
    division_name VARCHAR(255) UNIQUE NOT NULL,
    division_name_short VARCHAR(255) UNIQUE NOT NULL,
    sort_order INT UNIQUE NOT NULL CHECK (sort_order > 0),
    division_link VARCHAR(255) UNIQUE NOT NULL,
    ----------------------------------------------
    CONSTRAINT division_link_format CHECK (division_link ~ link_regex)
)
'''
CREATE_divisions = CREATE_divisions.replace('link_regex', '\'' + link_regex + '\'')
print(CREATE_divisions)

In [ ]:
with db.connect('saber.db') as con:
    con.begin()#begin transaction

    #drop current tables
    table_names = con.sql('show tables').pl()['name']

    for table in table_names:
        DROP = f'DROP TABLE IF EXISTS {table}'
        con.sql(DROP.format(table))

    try:
        #create tables
        con.sql(CREATE_sports)
        con.sql(CREATE_leagues)
        con.sql(CREATE_divisions)
    except:
        #rollback on error
        con.rollback()
        raise
    finally:
        #commit changes
        con.commit()

    print(con.sql('show tables').pl())